# Generate YOLO Bounding Box Annotations from FLAME 2 Thermal Images

This notebook creates bounding box annotations by thresholding the thermal channel.

**Strategy:**
1. Load thermal image
2. Convert to grayscale (the colormap encodes intensity)
3. Threshold to find hot regions
4. Find contours → bounding boxes
5. Filter small/noise boxes
6. Save in YOLO format: `class_id x_center y_center width height` (normalized)

In [ ]:
import os
from pathlib import Path
from typing import List, Tuple

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Paths
DATA_DIR = Path("../data/flame2")
THERMAL_DIR = DATA_DIR / "frames" / "254p Thermal Images"
RGB_DIR = DATA_DIR / "frames" / "254p RGB Images"
LABELS_FILE = DATA_DIR / "labels.txt"

print(f"Thermal dir exists: {THERMAL_DIR.exists()}")
print(f"RGB dir exists: {RGB_DIR.exists()}")
print(f"Labels file exists: {LABELS_FILE.exists()}")

## 1. Parse Labels File

In [ ]:
def parse_labels(labels_path: Path) -> dict:
    """
    Parse the FLAME 2 labels file.
    Returns dict mapping frame_number -> (has_fire, has_smoke)
    """
    labels = {}
    with open(labels_path) as f:
        for line in f:
            line = line.strip()
            # Skip header/comment lines
            if not line or line.startswith("First") or line.startswith("Fire") or line.startswith("Smoke"):
                continue
            
            parts = line.split()
            if len(parts) == 3:
                start, end, label = int(parts[0]), int(parts[1]), parts[2]
                has_fire = label[0] == 'Y'
                has_smoke = label[1] == 'Y'
                for frame_num in range(start, end + 1):
                    labels[frame_num] = (has_fire, has_smoke)
    return labels

frame_labels = parse_labels(LABELS_FILE)
print(f"Total labeled frames: {len(frame_labels)}")

# Count by class
fire_count = sum(1 for (has_fire, _) in frame_labels.values() if has_fire)
no_fire_count = len(frame_labels) - fire_count
print(f"Fire frames: {fire_count}")
print(f"No-fire frames: {no_fire_count}")

## 2. Thermal Thresholding Pipeline

In [ ]:
def extract_fire_boxes(
    thermal_path: Path,
    threshold: int = 200,
    min_area: int = 100,
    max_area: int = 50000
) -> List[Tuple[int, int, int, int]]:
    """
    Extract bounding boxes from thermal image.
    
    The thermal images use a colormap where bright (high intensity) = hot.
    We convert to grayscale and threshold to find hot regions.
    
    Returns list of (x, y, w, h) bounding boxes in pixel coordinates.
    """
    # Load image
    img = cv2.imread(str(thermal_path))
    if img is None:
        return []
    
    # Convert to grayscale (intensity from colormap)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Threshold - hot regions are bright
    _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    
    # Find contours
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    boxes = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if min_area <= area <= max_area:
            x, y, w, h = cv2.boundingRect(contour)
            boxes.append((x, y, w, h))
    
    return boxes


def boxes_to_yolo(
    boxes: List[Tuple[int, int, int, int]],
    img_width: int = 254,
    img_height: int = 254,
    class_id: int = 0  # 0 = fire
) -> List[str]:
    """
    Convert pixel bounding boxes to YOLO format.
    YOLO format: class_id x_center y_center width height (all normalized 0-1)
    """
    yolo_lines = []
    for (x, y, w, h) in boxes:
        x_center = (x + w / 2) / img_width
        y_center = (y + h / 2) / img_height
        w_norm = w / img_width
        h_norm = h / img_height
        yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")
    return yolo_lines

## 3. Visualize Pipeline on Sample Frames

In [ ]:
def visualize_detection(frame_num: int, threshold: int = 200):
    """Visualize the detection pipeline for a single frame."""
    thermal_path = THERMAL_DIR / f"254p Thermal Frame ({frame_num}).jpg"
    rgb_path = RGB_DIR / f"254p RGB Frame ({frame_num}).jpg"
    
    # Load images
    thermal = cv2.imread(str(thermal_path))
    rgb = cv2.imread(str(rgb_path))
    rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
    
    # Get boxes
    boxes = extract_fire_boxes(thermal_path, threshold=threshold)
    
    # Draw boxes on both images
    thermal_vis = thermal.copy()
    rgb_vis = rgb.copy()
    for (x, y, w, h) in boxes:
        cv2.rectangle(thermal_vis, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.rectangle(rgb_vis, (x, y), (x+w, y+h), (0, 255, 0), 2)
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(rgb)
    axes[0].set_title(f"RGB Frame {frame_num}")
    axes[0].axis("off")
    
    axes[1].imshow(cv2.cvtColor(thermal, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Thermal Frame {frame_num}")
    axes[1].axis("off")
    
    axes[2].imshow(cv2.cvtColor(thermal_vis, cv2.COLOR_BGR2RGB))
    axes[2].set_title(f"Detected Boxes ({len(boxes)} found)")
    axes[2].axis("off")
    
    label = frame_labels.get(frame_num, (False, False))
    fire_str = "Fire" if label[0] else "No Fire"
    smoke_str = "Smoke" if label[1] else "No Smoke"
    fig.suptitle(f"Frame {frame_num}: {fire_str}, {smoke_str}", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    return boxes

In [ ]:
# Test on a fire frame
boxes = visualize_detection(13701, threshold=200)

In [ ]:
# Test on a no-fire frame (should find no boxes)
boxes = visualize_detection(1000, threshold=200)

In [ ]:
# Test different threshold values
for thresh in [150, 180, 200, 220]:
    print(f"\nThreshold: {thresh}")
    boxes = visualize_detection(30000, threshold=thresh)

## 4. Generate Annotations for All Fire Frames

We'll only generate annotations for frames labeled as having fire.

In [ ]:
# This cell generates annotations - run after tuning threshold

THRESHOLD = 200  # Tune this based on visualization above
MIN_AREA = 50
MAX_AREA = 50000

OUTPUT_DIR = DATA_DIR / "annotations"
OUTPUT_DIR.mkdir(exist_ok=True)

# Get fire frames only
fire_frames = [fn for fn, (has_fire, _) in frame_labels.items() if has_fire]
print(f"Processing {len(fire_frames)} fire frames...")

stats = {"with_boxes": 0, "empty": 0, "errors": 0}

for i, frame_num in enumerate(fire_frames):
    if i % 5000 == 0:
        print(f"  Progress: {i}/{len(fire_frames)}")
    
    thermal_path = THERMAL_DIR / f"254p Thermal Frame ({frame_num}).jpg"
    
    try:
        boxes = extract_fire_boxes(thermal_path, threshold=THRESHOLD, min_area=MIN_AREA, max_area=MAX_AREA)
        yolo_lines = boxes_to_yolo(boxes)
        
        # Save annotation file
        ann_path = OUTPUT_DIR / f"frame_{frame_num}.txt"
        with open(ann_path, "w") as f:
            f.write("\n".join(yolo_lines))
        
        if boxes:
            stats["with_boxes"] += 1
        else:
            stats["empty"] += 1
    except Exception as e:
        stats["errors"] += 1

print(f"\nDone!")
print(f"  Frames with boxes: {stats['with_boxes']}")
print(f"  Empty (no hot spots): {stats['empty']}")
print(f"  Errors: {stats['errors']}")

## 5. Create YOLO Dataset Structure

YOLO expects:
```
dataset/
  images/
    train/
    val/
  labels/
    train/
    val/
  data.yaml
```

In [ ]:
# Create data.yaml for YOLO training
data_yaml = """
# FLAME 2 Fire Detection Dataset
path: ../data/flame2/yolo_dataset
train: images/train
val: images/val

names:
  0: fire
"""

print(data_yaml)